# Phase 5: select the NEDI–IMDN fusion weight

This notebook chooses **one fixed fusion percentage** before the final test.

- It uses DIV2K validation images 0801–0805, not Set5, Set14, BSD100, or Urban100.
- It uses the same fixed 360 × 360 centre area at x2, x3, and x4.
- It tries IMDN contributions from 0% to 100% in 10% steps.
- It selects the highest average PSNR-Y; SSIM-Y is the first tie-break.
- It does not train a model.

NEDI and IMDN are reconstructed once per image and cached in Drive. If Colab disconnects, rerun all cells; completed reconstructions are reused.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/divinesta/SuperResolution-Comparative-Analysis.git'
REPO_ROOT = Path('/content/SuperResolution-Comparative-Analysis')
if REPO_ROOT.exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt')],
    check=True,
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('Repository and dependencies ready.')


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Select Runtime > Change runtime type > T4 GPU.')

DEVICE = torch.device('cuda')
DATA_ROOT = Path('/content/drive/MyDrive/FYP_SR_Data')
CHECKPOINT_ROOT = DATA_ROOT / 'checkpoints'
VALIDATION_ROOT = DATA_ROOT / 'validation' / 'DIV2K_valid_HR_subset'
OUTPUT_ROOT = DATA_ROOT / 'results' / 'phase5' / 'weight_selection_global_v1'
CACHE_ROOT = OUTPUT_ROOT / 'reconstruction_cache'
METRICS_ROOT = OUTPUT_ROOT / 'metrics'
FIGURE_ROOT = OUTPUT_ROOT / 'figures'
for directory in (VALIDATION_ROOT, CACHE_ROOT, METRICS_ROOT, FIGURE_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

VALIDATION_IDS = ('0801', '0802', '0803', '0804', '0805')
SCALES = (2, 3, 4)
HR_CROP_SIZE = 360
print('GPU:', torch.cuda.get_device_name(DEVICE))
print('Output folder:', OUTPUT_ROOT)


## 1. Obtain the separate validation images

Only five HR images are kept in Google Drive. If they are missing, this cell downloads the official DIV2K validation archive to temporary Colab storage and extracts only images 0801–0805. The DIV2K data is provided for academic research by ETH Zurich.


In [ ]:
from urllib.request import urlretrieve
from zipfile import ZipFile

DIV2K_VALIDATION_URL = 'https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip'
missing_ids = [image_id for image_id in VALIDATION_IDS if not (VALIDATION_ROOT / f'{image_id}.png').is_file()]
if missing_ids:
    archive_path = Path('/content/DIV2K_valid_HR.zip')
    if not archive_path.is_file():
        print('Downloading the official DIV2K validation HR archive...')
        urlretrieve(DIV2K_VALIDATION_URL, archive_path)
    with ZipFile(archive_path) as archive:
        available = set(archive.namelist())
        for image_id in missing_ids:
            member = f'DIV2K_valid_HR/{image_id}.png'
            if member not in available:
                raise FileNotFoundError(f'{member} is missing from the downloaded archive.')
            with archive.open(member) as source, (VALIDATION_ROOT / f'{image_id}.png').open('wb') as target:
                target.write(source.read())
            print('Saved:', VALIDATION_ROOT / f'{image_id}.png')

validation_paths = [VALIDATION_ROOT / f'{image_id}.png' for image_id in VALIDATION_IDS]
if not all(path.is_file() for path in validation_paths):
    raise RuntimeError('The five required DIV2K validation images are not ready.')
print('Validation images ready:', [path.name for path in validation_paths])


## 2. Fixed percentage choices

An **IMDN weight of 0.9** means 90% IMDN and 10% NEDI. The two endpoints are important: 0.0 is NEDI alone and 1.0 is IMDN alone.


In [ ]:
from app.fusion.weighted import DEFAULT_IMDN_WEIGHTS

IMDN_WEIGHTS = DEFAULT_IMDN_WEIGHTS
print('Percentages to test:')
for imdn_weight in IMDN_WEIGHTS:
    print(f'  IMDN {imdn_weight:.0%} + NEDI {1.0 - imdn_weight:.0%}')


## 3. Reconstruct and cache the 15 validation cases

For each of the five images, the notebook uses the same 360 × 360 centre crop. It creates LR with the project's bicubic rule, then produces NEDI and IMDN outputs at x2, x3, and x4. Timing is not measured here because this step is only choosing image quality.


In [ ]:
import json
from datetime import UTC, datetime

from app.deep_learning.alignment import align_reconstruction_to_target
from app.deep_learning.checkpoints import IMDN_CHECKPOINTS, download_official_imdn_checkpoint
from app.deep_learning.imdn import imdn_upsample, load_pretrained_imdn
from app.evaluation.experiment import save_image, write_results_csv
from app.evaluation.images import bicubic_downsample, load_rgb_image
from app.fusion.weighted import evaluate_weight_grid
from app.traditional.nedi import NEDIConfig, nedi_upsample_rgb

NEDI_CONFIG = NEDIConfig(window_size=8, edge_threshold=8.0)
checkpoint_paths = {
    scale: download_official_imdn_checkpoint(CHECKPOINT_ROOT, scale)
    for scale in SCALES
}
print('All IMDN checkpoints are present and checksum-verified.')


def centre_crop(image, size):
    if image.width < size or image.height < size:
        raise ValueError(f'Image {image.size} is smaller than the required {size}x{size} crop.')
    left = (image.width - size) // 2
    top = (image.height - size) // 2
    return image.crop((left, top, left + size, top + size)), (left, top, left + size, top + size)


def load_cached_rgb(path, expected_size):
    if not path.is_file():
        return None
    image = load_rgb_image(path)
    if image.size != expected_size:
        raise ValueError(f'Cached image has wrong size: {path} is {image.size}, expected {expected_size}.')
    return image


all_records = []
checkpoint_csv = METRICS_ROOT / 'fusion_validation_all_weights.csv'
for scale in SCALES:
    model = load_pretrained_imdn(checkpoint_paths[scale], scale, DEVICE)
    for hr_path in validation_paths:
        source_hr = load_rgb_image(hr_path)
        crop_hr, crop_box = centre_crop(source_hr, HR_CROP_SIZE)
        reference_hr, lr_image = bicubic_downsample(crop_hr, scale)

        case_root = CACHE_ROOT / f'x{scale}' / hr_path.stem
        nedi_path = case_root / 'nedi.png'
        imdn_path = case_root / 'imdn.png'
        hr_cache_path = case_root / 'reference_hr.png'
        lr_cache_path = case_root / 'input_lr.png'

        nedi_image = load_cached_rgb(nedi_path, reference_hr.size)
        if nedi_image is None:
            print(f'Running NEDI x{scale} for {hr_path.name}...')
            nedi_image = nedi_upsample_rgb(
                lr_image, scale, reference_hr.size, NEDI_CONFIG
            ).image
            save_image(nedi_image, nedi_path)

        imdn_image = load_cached_rgb(imdn_path, reference_hr.size)
        if imdn_image is None:
            print(f'Running IMDN x{scale} for {hr_path.name}...')
            native_imdn = imdn_upsample(model, lr_image, DEVICE)
            imdn_image = align_reconstruction_to_target(native_imdn, reference_hr.size).image
            save_image(imdn_image, imdn_path)

        if not hr_cache_path.is_file():
            save_image(reference_hr, hr_cache_path)
        if not lr_cache_path.is_file():
            save_image(lr_image, lr_cache_path)

        weight_records = evaluate_weight_grid(
            reference_hr, nedi_image, imdn_image, scale, weights=IMDN_WEIGHTS
        )
        for record in weight_records:
            all_records.append({
                'validation_dataset': 'DIV2K_valid',
                'image': hr_path.name,
                'scale': f'x{scale}',
                'method': 'weighted_nedi_imdn_fusion',
                'crop_rule': 'fixed_360x360_centre_hr_crop',
                'crop_left': crop_box[0],
                'crop_top': crop_box[1],
                'crop_right': crop_box[2],
                'crop_bottom': crop_box[3],
                'degradation': 'project_bicubic_downsampling',
                'metric_border_pixels': scale,
                **record,
            })
        write_results_csv(all_records, checkpoint_csv, overwrite=True)
        print(f'Complete: {hr_path.name} x{scale}; saved {len(all_records)} metric rows.')

    del model
    torch.cuda.empty_cache()

expected_rows = len(VALIDATION_IDS) * len(SCALES) * len(IMDN_WEIGHTS)
if len(all_records) != expected_rows:
    raise RuntimeError(f'Expected {expected_rows} validation rows; produced {len(all_records)}.')
print('All validation cases complete:', len(all_records), 'rows')


## 4. Select one global weight

The selected percentage is the one with the highest mean PSNR-Y over all 15 image-scale cases. SSIM-Y is used only if PSNR-Y ties. The selected percentage will later be locked for the full benchmark run.


In [ ]:
from app.fusion.weighted import select_best_weight, summarise_weight_results

summary_records = summarise_weight_results(all_records)
summary_csv = write_results_csv(
    summary_records,
    METRICS_ROOT / 'fusion_weight_summary.csv',
    overwrite=True,
)
selected = select_best_weight(summary_records)
selection_record = {
    'method': 'weighted_nedi_imdn_fusion',
    'selected_imdn_weight': selected['imdn_weight'],
    'selected_nedi_weight': selected['nedi_weight'],
    'selection_rule': 'highest_mean_psnr_y_then_mean_ssim_y_then_larger_imdn_weight',
    'one_global_weight_for_all_scales': True,
    'validation_dataset': 'DIV2K_valid',
    'validation_image_ids': list(VALIDATION_IDS),
    'validation_crop_rule': 'fixed_360x360_centre_hr_crop',
    'validation_scales': list(SCALES),
    'candidate_imdn_weights': list(IMDN_WEIGHTS),
    'validation_sample_count_per_weight': selected['sample_count'],
    'selected_mean_psnr_y': selected['mean_psnr_y'],
    'selected_mean_ssim_y': selected['mean_ssim_y'],
    'imdn_checkpoint_sha256_by_scale': {
        f'x{scale}': IMDN_CHECKPOINTS[scale].sha256 for scale in SCALES
    },
    'div2k_source_url': DIV2K_VALIDATION_URL,
    'generated_at_utc': datetime.now(UTC).isoformat(),
}
selection_json = METRICS_ROOT / 'fusion_selected_weight.json'
selection_json.write_text(json.dumps(selection_record, indent=2) + '\n', encoding='utf-8')

print('\nSELECTED FUSION:')
print(f"  IMDN: {selected['imdn_weight']:.0%}")
print(f"  NEDI: {selected['nedi_weight']:.0%}")
print(f"  Validation PSNR-Y: {selected['mean_psnr_y']:.4f}")
print(f"  Validation SSIM-Y: {selected['mean_ssim_y']:.6f}")
print('Saved:', summary_csv)
print('Saved:', selection_json)


In [ ]:
import matplotlib.pyplot as plt

weights = [float(row['imdn_weight']) for row in summary_records]
psnr_y = [float(row['mean_psnr_y']) for row in summary_records]
ssim_y = [float(row['mean_ssim_y']) for row in summary_records]

figure, left_axis = plt.subplots(figsize=(8, 5))
right_axis = left_axis.twinx()
left_axis.plot(weights, psnr_y, marker='o', color='#1769aa', label='PSNR-Y')
right_axis.plot(weights, ssim_y, marker='s', color='#c44e52', label='SSIM-Y')
left_axis.axvline(float(selected['imdn_weight']), color='#333333', linestyle='--', label='Selected weight')
left_axis.set_xlabel('IMDN weight (NEDI weight = 1 − IMDN weight)')
left_axis.set_ylabel('Mean PSNR-Y (dB)', color='#1769aa')
right_axis.set_ylabel('Mean SSIM-Y', color='#c44e52')
left_axis.set_title('Phase 5 validation: weighted NEDI–IMDN fusion')
left_axis.grid(alpha=0.25)
lines = left_axis.get_lines() + right_axis.get_lines()
left_axis.legend(lines, [line.get_label() for line in lines], loc='best')
figure.tight_layout()
figure_path = FIGURE_ROOT / 'fusion_weight_ablation.png'
figure.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', figure_path)


## 5. Return the results

Download and send back these three files:

1. **fusion_validation_all_weights.csv**
2. **fusion_weight_summary.csv**
3. **fusion_selected_weight.json**

They are in **MyDrive/FYP_SR_Data/results/phase5/weight_selection_global_v1/metrics/**.

After those results are checked, the next notebook will use the selected percentage for the full Set5, Set14, BSD100, and Urban100 evaluation.
